# Hosting LangGraph agent with Amazon Bedrock models in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to host your existing agent, using Amazon Bedrock AgentCore Runtime. 

We will focus on a LangGraph with Amazon Bedrock model example. For Strands Agents with Amazon Bedrock model check [here](../01-strands-with-bedrock-model)
and for a Strands Agents with an OpenAI model check [here](../03-strands-with-openai-model).

### Tutorial Details

| Information         | Details                                                                      |
|:--------------------|:-----------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                               |
| Agent type          | Single                                                                       |
| Agentic Framework   | LangGraph                                                                    |
| LLM model           | Anthropic Claude Haiku 4.5                                                  |
| Tutorial components | Hosting agent on AgentCore Runtime. Using LangGraph and Amazon Bedrock Model |
| Tutorial vertical   | Cross-vertical                                                               |
| Example complexity  | Easy                                                                         |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                                 |

### Tutorial Architecture

In this tutorial we will describe how to deploy an existing agent to AgentCore runtime. 

For demonstration purposes, we will  use a LangGraph agent using Amazon Bedrock models

In our example we will use a very simple agent with two tools: `get_weather` and `get_time`. 

<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="50%"/>
</div>

### Tutorial Key Features

* Hosting Agents on Amazon Bedrock AgentCore Runtime
* Using Amazon Bedrock models
* Using LangGraph


## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* LangGraph
* Docker running

In [ ]:
!uv pip install --force-reinstall -U -r requirements.txt --quiet

## Creating your agents and experimenting locally

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

For production agentic applications we will need to decouple the agent creation process from the agent invocation one. With AgentCore Runtime, we will decorate the invocation part of our agent with the `@app.entrypoint` decorator and have it as the entry point for our runtime. Let's first look how each agent is developed during the experimentation phase.

The architecture here will look as following:

<div style="text-align:left">
    <img src="images/architecture_local.png" width="60%"/>
</div>

In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
user_name = os.getenv("USER_NAME")
if not user_name:
    raise ValueError("USER_NAME environment variable is not set. Please set it in the .env file.")


In [ ]:
%%writefile langgraph_tools.py
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
import argparse
import json
import operator
import math

# Create calculator tool
@tool
def calculator(expression: str) -> str:
    """
    Calculate the result of a mathematical expression.

    Args:
        expression: A mathematical expression as a string (e.g., "2 + 3 * 4", "sqrt(16)", "sin(pi/2)")

    Returns:
        The result of the calculation as a string
    """
    try:
        # Define safe functions that can be used in expressions
        safe_dict = {
            "__builtins__": {},
            "abs": abs, "round": round, "min": min, "max": max,
            "sum": sum, "pow": pow,
            # Math functions
            "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, "tan": math.tan,
            "log": math.log, "log10": math.log10, "exp": math.exp,
            "pi": math.pi, "e": math.e,
            "ceil": math.ceil, "floor": math.floor,
            "degrees": math.degrees, "radians": math.radians,
            # Basic operators (for explicit use)
            "add": operator.add, "sub": operator.sub,
            "mul": operator.mul, "truediv": operator.truediv,
        }

        # Evaluate the expression safely
        result = eval(expression, safe_dict)
        return str(result)

    except ZeroDivisionError:
        return "Error: Division by zero"
    except ValueError as e:
        return f"Error: Invalid value - {str(e)}"
    except SyntaxError:
        return "Error: Invalid mathematical expression"
    except Exception as e:
        return f"Error: {str(e)}"

# Create a custom weather tool
@tool
def weather():
    """Get weather"""  # Dummy implementation
    return "sunny"

# Define the agent using manual LangGraph construction
def create_agent():
    """Create and configure the LangGraph agent"""
    from langchain_aws import ChatBedrock

    # Initialize your LLM (adjust model and parameters as needed)
    llm = ChatBedrock(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # or your preferred model
        model_kwargs={"temperature": 0.1}
    )

    # Bind tools to the LLM
    tools = [calculator, weather]
    llm_with_tools = llm.bind_tools(tools)

    # System message
    system_message = "You're a helpful assistant. You can do simple math calculation, and tell the weather."

    # Define the chatbot node
    def chatbot(state: MessagesState):
        # Add system message if not already present
        messages = state["messages"]
        if not messages or not isinstance(messages[0], SystemMessage):
            messages = [SystemMessage(content=system_message)] + messages

        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}

    # Create the graph
    graph_builder = StateGraph(MessagesState)

    # Add nodes
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))

    # Add edges
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")

    # Set entry point
    graph_builder.set_entry_point("chatbot")

    # Compile the graph
    return graph_builder.compile()

# Initialize the agent
agent = create_agent()

def langgraph_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")

    # Create the input in the format expected by LangGraph
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})

    # Extract the final message content
    return response["messages"][-1].content

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = langgraph_bedrock(json.loads(args.payload))
    print(response)

Overwriting langgraph_tools.py


#### Invoking local agent

In [34]:
!uv run langgraph_tools.py '{"prompt": "What is the weather now?"}'

/Users/eric.fu/projects/xealth/agentcore-samples/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
The weather right now is **sunny**! ☀️ It's a nice day out there.


## Preparing your agent for deployment on AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to:
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

### LangGraph with Amazon Bedrock model
Let's start with our LangGraph using Amazon Bedrock model. Other examples with different frameworks and models are available in the parent directories

In [35]:
%%writefile langgraph_bedrock.py
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import argparse
import json
import operator
import math

app = BedrockAgentCoreApp()

# Create calculator tool
@tool
def calculator(expression: str) -> str:
    """
    Calculate the result of a mathematical expression.

    Args:
        expression: A mathematical expression as a string (e.g., "2 + 3 * 4", "sqrt(16)", "sin(pi/2)")

    Returns:
        The result of the calculation as a string
    """
    try:
        # Define safe functions that can be used in expressions
        safe_dict = {
            "__builtins__": {},
            "abs": abs, "round": round, "min": min, "max": max,
            "sum": sum, "pow": pow,
            # Math functions
            "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, "tan": math.tan,
            "log": math.log, "log10": math.log10, "exp": math.exp,
            "pi": math.pi, "e": math.e,
            "ceil": math.ceil, "floor": math.floor,
            "degrees": math.degrees, "radians": math.radians,
            # Basic operators (for explicit use)
            "add": operator.add, "sub": operator.sub,
            "mul": operator.mul, "truediv": operator.truediv,
        }

        # Evaluate the expression safely
        result = eval(expression, safe_dict)
        return str(result)

    except ZeroDivisionError:
        return "Error: Division by zero"
    except ValueError as e:
        return f"Error: Invalid value - {str(e)}"
    except SyntaxError:
        return "Error: Invalid mathematical expression"
    except Exception as e:
        return f"Error: {str(e)}"

# Create a custom weather tool
@tool
def weather():
    """Get weather"""  # Dummy implementation
    return "sunny"

# Define the agent using manual LangGraph construction
def create_agent():
    """Create and configure the LangGraph agent"""
    from langchain_aws import ChatBedrock

    # Initialize your LLM (adjust model and parameters as needed)
    llm = ChatBedrock(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # or your preferred model
        model_kwargs={"temperature": 0.1}
    )

    # Bind tools to the LLM
    tools = [calculator, weather]
    llm_with_tools = llm.bind_tools(tools)

    # System message
    system_message = "You're a helpful assistant. You can do simple math calculation, and tell the weather."

    # Define the chatbot node
    def chatbot(state: MessagesState):
        # Add system message if not already present
        messages = state["messages"]
        if not messages or not isinstance(messages[0], SystemMessage):
            messages = [SystemMessage(content=system_message)] + messages

        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}

    # Create the graph
    graph_builder = StateGraph(MessagesState)

    # Add nodes
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))

    # Add edges
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")

    # Set entry point
    graph_builder.set_entry_point("chatbot")

    # Compile the graph
    return graph_builder.compile()

# Initialize the agent
agent = create_agent()

@app.entrypoint
def langgraph_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")

    # Create the input in the format expected by LangGraph
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})

    # Extract the final message content
    return response["messages"][-1].content

if __name__ == "__main__":
    app.run()

Overwriting langgraph_bedrock.py


## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Configure AgentCore Runtime deployment

First we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<div style="text-align:left">
    <img src="images/configure.png" width="40%"/>
</div>

In [3]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import os

boto_session = Session(
    aws_access_key_id=os.environ.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.environ.get('AWS_SECRET_ACCESS_KEY'),
    aws_session_token=os.environ.get('AWS_SESSION_TOKEN'),
    region_name=os.environ.get('AWS_REGION')
)
region = boto_session.region_name

agentcore_runtime = Runtime()

user_name = os.environ.get('USER_NAME')
if not user_name:
    raise ValueError("USER_NAME environment variable is not set")

agent_name = "langgraph_claude_getting_started_" + user_name
response = agentcore_runtime.configure(
    entrypoint="langgraph_bedrock.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
)
response

Entrypoint parsed: file=/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/02-langgraph-with-bedrock-model/langgraph_bedrock.py, bedrock_agentcore_name=langgraph_bedrock
Configuring BedrockAgentCore agent: langgraph_claude_getting_started_eric_fu
Generated .dockerignore
Generated Dockerfile: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/02-langgraph-with-bedrock-model/Dockerfile
Generated .dockerignore: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/02-langgraph-with-bedrock-model/.dockerignore
Setting 'langgraph_claude_getting_started_eric_fu' as default agent
Bedrock AgentCore configured: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/02-langgraph-with-bedrock-model/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/02-langgraph-with-bedrock-model/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/02-langgraph-with-bedrock-model/Dockerfile'), dockerignore_path=PosixPath('/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/02-langgraph-with-bedrock-model/.dockerignore'), runtime='Docker', region='us-west-2', account_id='372080370602', execution_role=None, ecr_repository=None, auto_create_ecr=True)

### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [4]:
launch_result = agentcore_runtime.launch()

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'langgraph_claude_getting_started_eric_fu' to account 372080370602 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: langgraph_claude_getting_started_eric_fu
✅ ECR repository available: 372080370602.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-langgraph_claude_getting_started_eric_fu
Getting or creating execution role for agent: langgraph_claude_getting_started_eric_fu
Using AWS region: us-west-2, account ID: 372080370602
Role name: AmazonBedrockAgentCoreSDKRuntime-us-west-2-97

✅ Reusing existing ECR repository: 372080370602.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-langgraph_claude_getting_started_eric_fu


✅ Reusing existing execution role: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-97fcddd95d
✅ Execution role available: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-97fcddd95d
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: langgraph_claude_getting_started_eric_fu
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-97fcddd95d
Reusing existing CodeBuild execution role: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-97fcddd95d
Using .dockerignore with 44 patterns
Uploaded source to S3: langgraph_claude_getting_started_eric_fu/source.zip
Updated CodeBuild project: bedrock-agentcore-langgraph_claude_getting_started_eric_fu-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 8.5

### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [5]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

Retrieved Bedrock AgentCore status for: langgraph_claude_getting_started_eric_fu


'READY'

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [6]:
invoke_response = agentcore_runtime.invoke({"prompt": "How much is 2+2?"})
invoke_response

{'ResponseMetadata': {'RequestId': 'be2272ea-a005-4a93-ac78-d55790d22111',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 10 Apr 2026 00:42:03 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'be2272ea-a005-4a93-ac78-d55790d22111',
   'x-amzn-bedrock-agentcore-runtime-session-id': '92081250-c5e9-4582-8996-52c238b5e275'},
  'RetryAttempts': 0},
 'runtimeSessionId': '92081250-c5e9-4582-8996-52c238b5e275',
 'contentType': 'application/json',
 'statusCode': 200,
 'response': ['2 + 2 = **4**']}

### Processing invocation results

We can now process our invocation results to include it in an application

In [8]:
from IPython.display import Markdown, display
import json
response_text = invoke_response['response'][0]
display(Markdown(response_text))

2 + 2 = **4**

### Invoking AgentCore Runtime with boto3

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [9]:
import boto3
import json
from IPython.display import Markdown as IPyMarkdown, display

agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 2+2?"})
)

# Capture the runtime session ID for lifecycle management
runtime_session_id = boto3_response.get('runtimeSessionId')
print(f"Runtime Session ID: {runtime_session_id}")

if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    decoded = json.loads(events[0].decode("utf-8"))
    display(Markdown(decoded))

Runtime Session ID: 05f3bf0a-2331-4cd5-a45c-dece640e48af


2 + 2 = **4**

### Stopping a Session 

You'll want to stop individual sessions when they're no longer needed.
This releases the microVM resources for that session while keeping the runtime alive
for new sessions. Below we demonstrate `stop_runtime_session`.

In [10]:
# --- Inline Session Lifecycle Demo ---
# stop_runtime_session releases the microVM resources for this specific session while keeping the runtime alive for new sessions.


if runtime_session_id:
    agentcore_client.stop_runtime_session(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=runtime_session_id,
        qualifier='DEFAULT'
    )
    print(f"✅ Session '{runtime_session_id}' stopped — microVM resources released")
else:
    print("⚠️ No session ID available to stop")

✅ Session '05f3bf0a-2331-4cd5-a45c-dece640e48af' stopped — microVM resources released


### Lifecycle Configuration Demo 

Now let's demonstrate how to configure a runtime with a shorter idle timeout.
We'll create a second runtime with a 5-minute (300 second) idle timeout to show
how lifecycle configuration affects session behavior. Both runtimes will coexist.

In [11]:
# --- Lifecycle Configuration Demo ---
# In production, choose a timeout appropriate for your workload:
#   - Development/testing: 5-15 minutes
#   - Interactive sessions: 30-60 minutes
#   - Long-running workloads: adjust as needed
#

agentcore_runtime_short = Runtime()
agent_name_short = "langgraph_claude_short_timeout"

# Configure with shorter idle timeout
response_short = agentcore_runtime_short.configure(
    entrypoint="langgraph_bedrock.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name_short
)

# Launch the second runtime
launch_result_short = agentcore_runtime_short.launch()
print(f"Second runtime launched: {launch_result_short.agent_id}")

# Wait for it to be ready
status_response_short = agentcore_runtime_short.status()
status_short = status_response_short.endpoint['status']
while status_short not in ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']:
    time.sleep(10)
    status_response_short = agentcore_runtime_short.status()
    status_short = status_response_short.endpoint['status']
    print(f"Short timeout runtime status: {status_short}")

# Now update the runtime with shorter idle timeout using boto3
# UpdateAgentRuntime is a full-replacement API — we must re-supply all required fields.
# First, retrieve the current runtime configuration.
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
current_runtime = agentcore_control_client.get_agent_runtime(
    agentRuntimeId=launch_result_short.agent_id
)

update_response = agentcore_control_client.update_agent_runtime(
    agentRuntimeId=launch_result_short.agent_id,
    agentRuntimeArtifact=current_runtime['agentRuntimeArtifact'],
    roleArn=current_runtime['roleArn'],
    networkConfiguration=current_runtime['networkConfiguration'],
    lifecycleConfiguration={
        'idleRuntimeSessionTimeout': 300  # 5 minutes
    }
)
print(f"✅ Runtime updated with 5-minute idle timeout")

# Invoke the second runtime to verify it works
invoke_response_short = agentcore_runtime_short.invoke({"prompt": "What is 3+3?"})
print(f"Second runtime response: {invoke_response_short['response'][0]}")

Entrypoint parsed: file=/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/02-langgraph-with-bedrock-model/langgraph_bedrock.py, bedrock_agentcore_name=langgraph_bedrock
Configuring BedrockAgentCore agent: langgraph_claude_short_timeout
Generated Dockerfile: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/02-langgraph-with-bedrock-model/Dockerfile
Generated .dockerignore: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/02-langgraph-with-bedrock-model/.dockerignore
Changing default agent from 'langgraph_claude_getting_started_eric_fu' to 'langgraph_claude_short_timeout'
Bedrock AgentCore configured: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/02-langgraph-with-bedrock-model/.bedrock_agentcore.yaml
🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containe

Repository doesn't exist, creating new ECR repository: bedrock-agentcore-langgraph_claude_short_timeout


Getting or creating execution role for agent: langgraph_claude_short_timeout
Using AWS region: us-west-2, account ID: 372080370602
Role name: AmazonBedrockAgentCoreSDKRuntime-us-west-2-1fbb3df52b
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-west-2-1fbb3df52b
Starting execution role creation process for agent: langgraph_claude_short_timeout
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-west-2-1fbb3df52b
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-west-2-1fbb3df52b
✓ Role created: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-1fbb3df52b
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-langgraph_claude_short_timeout
Role creation complete and ready for use with Bedrock AgentCore
✅ Execution role available: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-1fbb3df52b
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role 

Second runtime launched: langgraph_claude_short_timeout-MRToXkH8UQ


Retrieved Bedrock AgentCore status for: langgraph_claude_short_timeout


✅ Runtime updated with 5-minute idle timeout
Second runtime response: 3 + 3 = **6**


## Cleanup

Let's now clean up the AgentCore Runtime and associated resources. We delete the runtime first to avoid undesired costs, then clean up supporting resources like ECR repositories.

In [12]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

('372080370602.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-langgraph_claude_getting_started_eric_fu',
 'langgraph_claude_getting_started_eric_fu-ku0GnT4kbI',
 'bedrock-agentcore-langgraph_claude_getting_started_eric_fu')

In [13]:
# --- Stop active sessions to release microVM resources ---
import boto3

agentcore_client = boto3.client('bedrock-agentcore', region_name=region)
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
ecr_client = boto3.client('ecr', region_name=region)

# Stop the active session to release its microVM resources
# In production, this is how you end individual user sessions while keeping the runtime alive
# AgentCore Runtime costs are based on vCPU and Memory — stopping sessions avoids undesired costs
# Note: If the session was already stopped in the earlier demo cell, this will raise a
# ResourceNotFoundException — the except block handles that gracefully.
if 'runtime_session_id' in locals() and runtime_session_id:
    try:
        agentcore_client.stop_runtime_session(
            agentRuntimeArn=launch_result.agent_arn,
            runtimeSessionId=runtime_session_id,
            qualifier='DEFAULT'
        )
        print(f"✅ Session '{runtime_session_id}' stopped")
    except Exception as e:
        print(f"⚠️ Failed to stop session '{runtime_session_id}': {e}")

# --- Delete both runtimes ---
# Original runtime
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )
    print(f"✅ Original runtime '{launch_result.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete original runtime: {e}")

# Short-timeout runtime
if 'launch_result_short' in locals():
    try:
        agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result_short.agent_id,
        )
        print(f"✅ Short-timeout runtime '{launch_result_short.agent_id}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete short-timeout runtime: {e}")

# --- Delete ECR repositories ---
try:
    ecr_client.delete_repository(
        repositoryName=launch_result.ecr_uri.split('/')[1],
        force=True
    )
    print(f"✅ ECR repository '{launch_result.ecr_uri.split('/')[1]}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete ECR repository: {e}")

if 'launch_result_short' in locals():
    try:
        ecr_client.delete_repository(
            repositoryName=launch_result_short.ecr_uri.split('/')[1],
            force=True
        )
        print(f"✅ Second ECR repository '{launch_result_short.ecr_uri.split('/')[1]}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete second ECR repository: {e}")

⚠️ Failed to stop session '05f3bf0a-2331-4cd5-a45c-dece640e48af': An error occurred (ResourceNotFoundException) when calling the StopRuntimeSession operation: Session 05f3bf0a-2331-4cd5-a45c-dece640e48af not found or has been terminated
✅ Original runtime 'langgraph_claude_getting_started_eric_fu-ku0GnT4kbI' deleted
✅ Short-timeout runtime 'langgraph_claude_short_timeout-MRToXkH8UQ' deleted
✅ ECR repository 'bedrock-agentcore-langgraph_claude_getting_started_eric_fu' deleted
✅ Second ECR repository 'bedrock-agentcore-langgraph_claude_short_timeout' deleted


Delete the CodeBuild projects (there are two of them)

# Congratulations!